# 04 — Parse Self-Reported Explanations (interactive)

Normalises `referred_words` and `referred_image_concepts` from the inference cache into per-mode JSON.

For grid runs use `run_parse_explanations.py` and `run_all.sh`.
Outputs → `data/vlm_explanations/<mode>/self_reported.json`


In [1]:
CONFIG = {
    "dataset_path": "../../datasets/translated/",
    "language": "uk",
    "inference_cache_dir": "./data/inference_cache",
    "lrp_results_dir": "./data/lrp_results",
    "output_dir": "./data/vlm_explanations",
    "modes": ["text", "image", "image+text"],
    "visual_modes": ["image", "image+text"],  # modes with LRP heatmaps
    "gpt_model": "gpt-5.1",
    "max_concurrent": 20,
    "seed": 42,
}

In [2]:
import asyncio
import base64
import json
import os
import re
import traceback
from pathlib import Path

from json_repair import repair_json
from openai import AsyncOpenAI
from tqdm import tqdm

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY environment variable must be set.")
OUT_DIR = Path(CONFIG["output_dir"])
CACHE_DIR = Path(CONFIG["inference_cache_dir"])
LRP_DIR = Path(CONFIG["lrp_results_dir"])

client = AsyncOpenAI(api_key=api_key)

LANG_NOTE = {
    "uk": "The meme is in Ukrainian. Returned concepts should be in Ukrainian.",
    "en": "The meme is in English. Returned concepts should be in English.",
}

In [3]:
def normalize_word(word):
    word = re.sub(r"^\W+|\W+$", "", str(word), flags=re.UNICODE)
    return word.lower().strip()

STOPWORDS_EN = {"a", "an", "the", "of", "in", "on", "at", "to", "with", "and", "or",
                "is", "are", "was", "be", "this", "that", "it", "its", "by", "for"}
STOPWORDS_UK = {
    'я', 'ти', 'ви', 'він', 'вона', 'вони', 'ми', 'це', 'те', 'ті',
    'мене', 'мені', 'мій', 'моя', 'себе', 'твоя', 'вас', 'вам',
    'їх', 'наші', 'ваші', 'всі', 'усі', 'всіх', 'тими', 'ній',
    'не', 'ні', 'за', 'на', 'що', 'так', 'але', 'та', 'бо', 'і',
    'щоб', 'аби', 'якщо', 'коли', 'чи', 'то', 'ще', 'вже', 'теж',
    'від', 'для', 'до', 'в', 'у', 'з', 'по', 'без', 'через', 'перед',
    'е', 'с', 'нам', 'наше',
}
STOPWORDS = {' '} #STOPWORDS_EN | STOPWORDS_UK

def clean_word_list(words, max_k=10):
    """Normalize, deduplicate, remove stopwords, return list of ≤max_k words."""
    seen, out = set(), []
    for w in (words or []):
        n = normalize_word(w)
        if n and n not in STOPWORDS and n not in seen and len(n) > 1:
            seen.add(n)
            out.append(n)
        if len(out) >= max_k:
            break
    return out

In [4]:
def parse_self_reported(mode):
    mode_key = mode.replace("+", "_")
    cache_dir = CACHE_DIR / mode_key
    out_dir = OUT_DIR / mode_key
    out_dir.mkdir(parents=True, exist_ok=True)

    manifest = json.load(open(cache_dir / "manifest.json"))
    results = []
    empty_count = 0

    for entry in tqdm(manifest, desc=f"Parse self-report [{mode}]"):
        record = json.load(open(entry["path"]))

        referred_words = clean_word_list(record.get("referred_words", []))
        referred_concepts = clean_word_list(record.get("referred_image_concepts", []))

        if not referred_words and not referred_concepts:
            empty_count += 1

        rec = {
            "id": record["id"],
            "mode": mode,
            "gold_labels": record.get("gold_labels", []),
            "pred_labels": record.get("pred_labels", []),
            "referred_words_raw": record.get("referred_words", []),
            "referred_words": referred_words,
            "referred_image_concepts_raw": record.get("referred_image_concepts", []),
            "referred_image_concepts": referred_concepts,
            "parse_ok": record.get("parse_ok", False),
        }
        results.append(rec)

    out_file = out_dir / "self_reported.json"
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"[{mode}] {len(results)} records saved, {empty_count} with empty explanations")
    return results

for mode in CONFIG["modes"]:
    parse_self_reported(mode)

Parse self-report [text]: 100%|██████████| 158/158 [00:00<00:00, 9375.66it/s]


[text] 158 records saved, 30 with empty explanations


Parse self-report [image]: 100%|██████████| 158/158 [00:00<00:00, 3042.10it/s]


[image] 158 records saved, 21 with empty explanations


Parse self-report [image+text]: 100%|██████████| 158/158 [00:00<00:00, 8066.26it/s]

[image+text] 158 records saved, 0 with empty explanations
